# Creación del Pipeline y exportación para Streamlit
Este cuaderno ('notebook') muestra cómo crear un `Pipeline` completo con `scikit-learn` (procesando tanto variables numéricas como categóricas), entrenar el modelo y luego guardarlo usando `joblib`.
Usamos un pequeño DataFrame simulado del problema 'Suscripción de Producto Bancario' para evitar llamadas de red que cuelguen el entorno al descargar los datos completos.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Creamos un dataset ficticio de prueba
data = {
    'age': [30, 40, 25, 35, 45, 22, 55, 33, 41, 28]*10,
    'balance': [1500, 200, 3000, 0, 500, 100, 10000, 50, 400, 2500]*10,
    'job': ['admin', 'blue-collar', 'student', 'services', 'management', 'student', 'management', 'services', 'blue-collar', 'admin']*10,
    'marital': ['married', 'married', 'single', 'divorced', 'married', 'single', 'married', 'single', 'divorced', 'single']*10,
    'subscribed': [1, 0, 1, 0, 1, 0, 1, 0, 0, 1]*10
}
df = pd.DataFrame(data)

# Definir variables predictoras y variable objetivo
X_df = df.drop('subscribed', axis=1)
y = df['subscribed']

# Identificar variables numéricas y categóricas
num_cols = X_df.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"Columnas numéricas: {num_cols}")
print(f"Columnas categóricas: {cat_cols}")


In [ ]:
# Definir preprocesamiento para columnas numéricas (imputación + escalado)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Definir preprocesamiento para columnas categóricas (imputación + codificación one-hot)
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar pasos de preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# Crear el pipeline completo combinando preprocesador y modelo clasificador
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Entrenar el pipeline
final_pipeline.fit(X_df, y)
print("Pipeline entrenado con éxito.")


### Metadatos de Variables para Generación de la Interfaz Web
Streamlit requiere información sobre las variables para crear automáticamente los controles de entrada (valores mínimos, máximos, y valores por defecto para entradas numéricas; listas desplegables para entradas categóricas). 
Podemos empaquetar esta información junto con el modelo en un único archivo.

In [ ]:
# Crear un diccionario con los metadatos para construir la interfaz fácilmente
feature_metadata = {}

for col in num_cols:
    s = X_df[col].dropna()
    feature_metadata[col] = {
        "type": "numerical",
        "min": float(s.min()),
        "max": float(s.max()),
        "median": float(s.median())
    }

for col in cat_cols:
    s = X_df[col].dropna()
    feature_metadata[col] = {
        "type": "categorical",
        "options": sorted(s.astype(str).unique().tolist())
    }

# Empaquetar tanto el pipeline como los metadatos resultantes
pack = {
    "pipeline": final_pipeline,
    "feature_metadata": feature_metadata,
    "classes_": final_pipeline.classes_.tolist()
}


In [ ]:
from joblib import dump
dump(pack, "modelo_produccion.joblib")
print("Guardado 'modelo_produccion.joblib'. ¡Ahora puedes usarlo en la app de Streamlit!")
